# Dataset creation

## Import

In [ ]:
import pickle
import numpy as np
from datasets import load_dataset

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

import warnings
warnings.filterwarnings("ignore")

Using device: cuda


`FEATURES` N x D matrix:
- bankName (int)
- phoneChanges (int)
- daysInBank (int)

`TRANSACTIONS` S x M_s x D matrix:
- from (int)
- to (int)
- amount (float32)
- type (int)
- isSAR (int)
- alertID (int)
- modelType (int)


node mapping: `f(n) = n + 2`:
- Source node (Integration): `-2` --> 0
- Sink node (Placement): `-1` --> 1

Type mapping:
- INITALBALANCE: `0`
- CASH: `1`
- TRANSFER: `2`

In [ ]:
def save_aml_dataset():
    ds = load_dataset("AMLGentex/easy")
    print(ds['train'].column_names)

    nodes_data = [[] for _ in range(1000000)] # N x D list of lists 
    bank_map, idx_bank = {}, 0
    transactions = [[] for _ in range(1000)]
    map_type = {
        'INITALBALANCE': 0,
        'CASH': 1,
        'TRANSFER': 2,
    }

    for r in ds['train']:
        id = r['nameOrig'] + 2
        if r['bankOrig'] not in bank_map:
            bank_map[r['bankOrig']] = idx_bank
            idx_bank += 1
        nodes_data[id] = [bank_map[r['bankOrig']], int(r['phoneChangesOrig']), int(r['daysInBankOrig'])]

        id = r['nameDest'] + 2
        if r['bankDest'] not in bank_map:
            bank_map[r['bankDest']] = idx_bank
            idx_bank += 1
        nodes_data[id] = [bank_map[r['bankDest']], int(r['phoneChangesDest']), int(r['daysInBankDest'])]

        transactions[r['step']].append([
            r['nameOrig'] + 2,
            r['nameDest'] + 2,
            float(r['amount']),
            map_type[r['type']],
            int(r['isSAR']),
            int(r['alertID']),
            int(r['modelType'])
        ])


    while nodes_data[-1] == []: nodes_data.pop()
    features = torch.tensor(nodes_data, dtype=torch.float32, device=device)
    torch.save(features, "../data/aml/features.pt")

    while transactions[-1] == []: transactions.pop()
    with open("../data/aml/transactions.pkl", "wb") as f:
        pickle.dump(transactions, f)

# save_aml_dataset() # ~23 mins